In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import Required Libraries

In [2]:
!pip install av --quiet

In [4]:
import logging
import os
import random
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_video
from tqdm import tqdm

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()]
)

logger = logging.getLogger('CRNN_UCF101')

In [ ]:
# Set random seeds for reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# Config

In [ ]:
class Config:
    data_dir = '/kaggle/input/ucf101/UCF101/UCF-101'
    output_dir = './output'
    batch_size = 8
    num_epochs = 50
    learning_rate = 0.001
    sequence_length = 16
    test_size = 0.2
    val_size = 0.1
    random_seed = 42
    device = 'cuda'
    samples_per_class = None

# Data Preprocessing and Loading

In [ ]:
def load_ucf101_data(
    data_dir: str, samples_per_class: Optional[int] = None
) -> Tuple[List[str], List[int], Dict[str, int]]:
    """Load UCF101 dataset from directory structure."""
    data_path = Path(data_dir)
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset directory not found: {data_dir}")

    video_paths = []
    labels = []
    class_names = []

    # Get all class directories
    class_dirs = [d for d in data_path.iterdir() if d.is_dir()]
    class_dirs.sort() # Ensure consistent ordering

    # Create class to index mapping
    class_to_idx = {cls_dir.name: idx for idx, cls_dir in enumerate(class_dirs)}

    print(f"Found {len(class_dirs)} classes in dataset")

    # Load videos from each class directory
    for class_dir in class_dirs:
        class_name = class_dir.name
        class_idx = class_to_idx[class_name]

        # Find all video files in the class directory
        video_files = []
        for ext in ["*.avi", "*.mp4", "*.mov", "*.mkv"]:
            video_files.extend(class_dir.glob(ext))


        # If samples_per_class is specified, take that many samples
        if samples_per_class is not None:
            if len(video_files) > samples_per_class:
                video_files = random.sample(video_files, samples_per_class)

        print(f"Class '{class_name}': {len(video_files)} videos")

        for video_file in video_files:
            video_paths.append(str(video_file))
            labels.append(class_idx)
            class_names.append(class_name)

    print(f"Total videos loaded: {len(video_paths)}")
    return video_paths, labels, class_to_idx

In [ ]:
# Create output directory
output_path = Path(Config.output_dir)
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load dataset
print("Loading UCF101 dataset...")
video_paths, labels, class_to_idx = load_ucf101_data(Config.data_dir, samples_per_class=15)
class_names = list(class_to_idx.keys())
num_classes = len(class_names)

Loading UCF101 dataset...
Found 101 classes in dataset
Class 'ApplyEyeMakeup': 15 videos
Class 'ApplyLipstick': 15 videos
Class 'Archery': 15 videos
Class 'BabyCrawling': 15 videos
Class 'BalanceBeam': 15 videos
Class 'BandMarching': 15 videos
Class 'BaseballPitch': 15 videos
Class 'Basketball': 15 videos
Class 'BasketballDunk': 15 videos
Class 'BenchPress': 15 videos
Class 'Biking': 15 videos
Class 'Billiards': 15 videos
Class 'BlowDryHair': 15 videos
Class 'BlowingCandles': 15 videos
Class 'BodyWeightSquats': 15 videos
Class 'Bowling': 15 videos
Class 'BoxingPunchingBag': 15 videos
Class 'BoxingSpeedBag': 15 videos
Class 'BreastStroke': 15 videos
Class 'BrushingTeeth': 15 videos
Class 'CleanAndJerk': 15 videos
Class 'CliffDiving': 15 videos
Class 'CricketBowling': 15 videos
Class 'CricketShot': 15 videos
Class 'CuttingInKitchen': 15 videos
Class 'Diving': 15 videos
Class 'Drumming': 15 videos
Class 'Fencing': 15 videos
Class 'FieldHockeyPenalty': 15 videos
Class 'FloorGymnastics': 15

In [ ]:
# Split data into train/test
train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    labels,
    test_size=Config.test_size,
    stratify=labels,
    random_state=Config.random_seed
)

In [ ]:
# Split training data into train/validation
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths,
    train_labels,
    test_size=Config.val_size,
    stratify=train_labels,
    random_state=Config.random_seed,
)

In [ ]:
print(
    f"Dataset split - Train: {len(train_paths)}, "
    f"Val: {len(val_paths)}, Test: {len(test_paths)}"
)

Dataset split - Train: 1090, Val: 122, Test: 303


In [ ]:
def create_data_transforms() -> Tuple[transforms.Compose, transforms.Compose]:
    """Create data transformations for training and validation."""
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform

In [ ]:
# Create data transforms
train_transform, val_transform = create_data_transforms()

In [ ]:
class VideoDataset(Dataset):
    """Custom dataset class for loading and preprocessing video data."""
    def __init__(
        self,
        video_paths: List[str],
        labels: List[int],
        sequence_length: int = 16,
        transform: Optional[transforms.Compose] = None,
        target_size: Tuple[int, int] = (112, 112)
    ) -> None:
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.transform = transform
        self.target_size = target_size

        if len(video_paths) != len(labels):
            raise ValueError("Number of video paths must match number of labels")

    def __len__(self) -> int:
        return len(self.video_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        try:
            # Load video using torchvision
            video_path = self.video_paths[idx]
            video, _, _ = read_video(video_path, pts_unit='sec')

            # Convert from (T, H, W, C) to (T, C, H, W)
            video = video.permute(0, 3, 1, 2).float() / 255.0

            # Sample frames uniformly if video is longer than sequence_length
            if video.shape[0] > self.sequence_length:
                indices = torch.linspace(
                    0, video.shape[0] - 1, self.sequence_length
                ).long()
                video = video[indices]
            elif video.shape[0] < self.sequence_length:
                # Repeat frames if video is shorter than sequence_length
                repeat_factor = self.sequence_length // video.shape[0] + 1
                video = video.repeat(repeat_factor, 1, 1, 1)[:self.sequence_length]

            # Resize frames to target size
            video = torch.nn.functional.interpolate(
                video, size=self.target_size, mode='bilinear', align_corners=False
            )

            # Apply transformations if provided
            if self.transform:
                transformed_frames = []
                for frame in video:
                    transformed_frame = self.transform(frame)
                    transformed_frames.append(transformed_frame)

                video = torch.stack(transformed_frames)

            label = self.labels[idx]
            return video, label

        except Exception as e:
            print(f"Error loading video {self.video_paths[idx]}: {str(e)}")
            # Return a dummy tensor and label in case of error
            dummy_video = torch.zeros(self.sequence_length, 3, *self.target_size)
            return dummy_video, self.labels[idx]

In [ ]:
# Create datasets
train_dataset = VideoDataset(train_paths, train_labels, Config.sequence_length, train_transform)
train_dataset.class_to_idx = class_to_idx # Save class mapping
val_dataset = VideoDataset(val_paths, val_labels, Config.sequence_length, val_transform)
test_dataset = VideoDataset(test_paths, test_labels, Config.sequence_length, val_transform)

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=Config.batch_size, shuffle=False, num_workers=2)

# Model Architecture

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """CNN component of CRNN for spatial feature extraction from video frames."""

    def __init__(
        self, input_channels: int = 3, hidden_dim: int = 512, dropout_rate: float = 0.3
    ) -> None:
        super(CNNFeatureExtractor, self).__init__()

        # First convolutional block
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        # Residual blocks
        self.res_block1 = self._make_residual_block(64, 128, stride=2)
        self.res_block2 = self._make_residual_block(128, 256, stride=2)
        self.res_block3 = self._make_residual_block(256, 512, stride=2)

        # Global average pooling and final layers
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(512, hidden_dim)

    def _make_residual_block(
        self, in_channels: int, out_channels: int, stride: int = 1
    ) -> nn.Sequential:
        """Create a residual block with skip connection."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of CNN feature extractor."""
        x = self.conv1(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.res_block3(x)

        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(x)
        x = self.fc(x)

        return x

In [ ]:
class RNNTemporalModel(nn.Module):
    """RNN component of CRNN for temporal sequence modeling."""

    def __init__(
        self,
        input_dim: int = 512,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout_rate: float = 0.3,
    ) -> None:
        super(RNNTemporalModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0,
            bidirectional=True,
        )

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1)
        )

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of RNN temporal model."""
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)  # (batch_size, seq_len, hidden_dim * 2)

        # Apply attention mechanism
        attention_weights = self.attention(lstm_out)  # (batch_size, seq_len, 1)
        attention_weights = F.softmax(attention_weights, dim=1)

        # Weighted sum of LSTM outputs
        attended_output = torch.sum(
            lstm_out * attention_weights, dim=1
        )  # (batch_size, hidden_dim * 2)

        return self.dropout(attended_output)

In [ ]:
class CRNN(nn.Module):
    """
    Complete CRNN model combining CNN feature extraction and RNN temporal modeling
    for video action recognition.
    """

    def __init__(
        self,
        num_classes: int,
        sequence_length: int = 16,
        input_channels: int = 3,
        cnn_hidden_dim: int = 512,
        rnn_hidden_dim: int = 256,
        rnn_num_layers: int = 2,
        dropout_rate: float = 0.3,
    ) -> None:
        super(CRNN, self).__init__()

        self.num_classes = num_classes
        self.sequence_length = sequence_length

        # CNN for spatial feature extraction
        self.cnn = CNNFeatureExtractor(
            input_channels=input_channels,
            hidden_dim=cnn_hidden_dim,
            dropout_rate=dropout_rate,
        )

        # RNN for temporal modeling
        self.rnn = RNNTemporalModel(
            input_dim=cnn_hidden_dim,
            hidden_dim=rnn_hidden_dim,
            num_layers=rnn_num_layers,
            dropout_rate=dropout_rate,
        )

        # Final classification layer
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden_dim * 2, rnn_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(rnn_hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of CRNN model."""
        batch_size, seq_len, channels, height, width = x.size()

        # Reshape for CNN processing: (batch_size * seq_len, channels, height, width)
        x = x.view(batch_size * seq_len, channels, height, width)

        # Extract spatial features using CNN
        cnn_features = self.cnn(x)  # (batch_size * seq_len, cnn_hidden_dim)

        # Reshape back for RNN: (batch_size, seq_len, cnn_hidden_dim)
        cnn_features = cnn_features.view(batch_size, seq_len, -1)

        # Model temporal dependencies using RNN
        rnn_features = self.rnn(cnn_features)  # (batch_size, rnn_hidden_dim * 2)

        # Final classification
        logits = self.classifier(rnn_features)  # (batch_size, num_classes)

        return logits

    def get_feature_extractor(self) -> nn.Module:
        """Get the CNN feature extractor for transfer learning or feature extraction."""
        return self.cnn

    def freeze_cnn(self) -> None:
        """Freeze CNN parameters for fine-tuning only the RNN and classifier."""
        for param in self.cnn.parameters():
            param.requires_grad = False

    def unfreeze_cnn(self) -> None:
        """Unfreeze CNN parameters for end-to-end training."""
        for param in self.cnn.parameters():
            param.requires_grad = True

In [ ]:
# Initialize model
model = CRNN(num_classes=num_classes, sequence_length=Config.sequence_length).to(device)
print(f"Model initialized with {num_classes} classes")

Model initialized with 101 classes


# Training

In [ ]:
def train_model(
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    num_epochs: int,
    learning_rate: float,
    save_path: str,
) -> Dict[str, List[float]]:
    """Train the CRNN model."""
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    logger.info("Starting training...")

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_pbar = tqdm(
            train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False
        )

        for batch_idx, (videos, labels) in enumerate(train_pbar):
            try:
                videos, labels = videos.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(videos)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()

                # Update progress bar
                train_pbar.set_postfix(
                    {
                        "Loss": f"{loss.item():.4f}",
                        "Acc": f"{100.0 * train_correct / train_total:.2f}%",
                    }
                )

            except Exception as e:
                logger.error(f"Error in training batch {batch_idx}: {str(e)}")
                continue

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        val_pbar = tqdm(
            val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False
        )

        with torch.no_grad():
            for batch_idx, (videos, labels) in enumerate(val_pbar):
                try:
                    videos, labels = videos.to(device), labels.to(device)
                    outputs = model(videos)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

                    # Update progress bar
                    val_pbar.set_postfix(
                        {
                            "Loss": f"{loss.item():.4f}",
                            "Acc": f"{100.0 * val_correct / val_total:.2f}%",
                        }
                    )

                except Exception as e:
                    logger.error(f"Error in validation batch {batch_idx}: {str(e)}")
                    continue

        # Calculate epoch metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100.0 * train_correct / train_total
        val_loss = val_loss / len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        # Update history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Print epoch results
        logger.info(
            f"Epoch {epoch + 1}/{num_epochs} - "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_acc": val_acc,
                    "class_to_idx": train_loader.dataset.class_to_idx
                    if hasattr(train_loader.dataset, "class_to_idx")
                    else None,
                },
                save_path,
            )
            logger.info(
                f"New best model saved with validation accuracy: {val_acc:.2f}%"
            )

        scheduler.step()

    logger.info(f"Training completed. Best validation accuracy: {best_val_acc:.2f}%")
    return history

In [ ]:
# Train model
model_save_path = output_path / "best_model.pth"
history = train_model(
    model,
    train_loader,
    val_loader,
    device,
    5, # Config.num_epochs,
    Config.learning_rate,
    str(model_save_path)
)

In [ ]:
# Load best model for evaluation
checkpoint = torch.load(model_save_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate

In [ ]:
def evaluate_model(
    model: torch.nn.Module,
    test_loader: DataLoader,
    device: torch.device,
    class_names: List[str],
) -> Dict[str, Any]:
    """Evaluate the trained model on test data."""
    model.eval()
    all_predictions = []
    all_labels = []

    print("Evaluating model...")

    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for videos, labels in test_pbar:
            try:
                videos, labels = videos.to(device), labels.to(device)
                outputs = model(videos)
                _, predicted = torch.max(outputs, 1)

                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            except Exception as e:
                print(f"Error in evaluation: {str(e)}")
                continue

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    report = classification_report(
        all_labels, all_predictions, target_names=class_names, output_dict=True
    )

    print(f"Test Accuracy: {accuracy:.4f}")

    return {
        "accuracy": accuracy,
        "classification_report": report,
        "predictions": all_predictions,
        "labels": all_labels,
    }

In [ ]:
# Evaluate on test set
test_results = evaluate_model(model, test_loader, device, class_names)

# Test

In [ ]:
def test_video_with_model(
    model_path: str,
    video_path: str,
    sequence_length: int = 16,
    target_size: Tuple[int, int] = (112, 112),
    device: str = "auto",
) -> None:
    """Test a trained model with an external video file."""
    # Load the model
    if device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    checkpoint = torch.load(model_path, map_location=device)
    class_to_idx = checkpoint.get("class_to_idx")
    if class_to_idx is None:
        raise ValueError("Model checkpoint does not contain class_to_idx mapping")

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    num_classes = len(idx_to_class)

    # Initialize model
    model = CRNN(num_classes=num_classes, sequence_length=sequence_length).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Create transform
    transform = transforms.Compose(
        [transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
    )

    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")

    # Initialize frame buffer
    frame_buffer = []
    frame_count = 0

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Create window
    cv2.namedWindow("Video Classification", cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess frame
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_tensor = torch.from_numpy(frame_rgb).permute(2, 0, 1).float() / 255.0
        frame_tensor = torch.nn.functional.interpolate(
            frame_tensor.unsqueeze(0),
            size=target_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        # Add to frame buffer
        frame_buffer.append(frame_tensor)
        if len(frame_buffer) > sequence_length:
            frame_buffer.pop(0)

        # When we have enough frames, make a prediction
        if len(frame_buffer) == sequence_length:
            # Stack frames and apply transform
            video_tensor = torch.stack(frame_buffer)
            video_tensor = transform(video_tensor).unsqueeze(0).to(device)

            # Make prediction
            with torch.no_grad():
                outputs = model(video_tensor)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                pred_prob, pred_class = torch.max(probs, 1)

            # Get class name and confidence
            class_name = idx_to_class.get(pred_class.item(), "Unknown")
            confidence = pred_prob.item()

            # Display prediction on frame
            label = f"{class_name} ({confidence:.2f})"
            cv2.putText(
                frame, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2
            )

        # Display frame
        cv2.imshow("Video Classification", frame)

        # Check for quit key
        if cv2.waitKey(int(1000 / fps)) & 0xFF == ord("q"):
            break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
#test_video_with_model(model_path, video_path, sequence_length=16, target_size=(112, 112), device='auto')